# 3. Organoid–Cell Relationship

## Purpose
This notebook assigns each single cell and nucleocentric object to its parent organoid,
then computes spatial relationship features (Euclidean distance, Mahalanobis distance,
and shell classification) for each cell relative to its parent organoid centroid.

This is **step 3 of Stage 4 (image-based profiling)**. It runs once per well-FOV and
is typically submitted as a child job via the SLURM scheduler.

## Inputs
Parquet files from `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/`:
- `sc_profiles_{well_fov}.parquet` — merged Nuclei + Cell + Cytoplasm features (required)
- `organoid_profiles_{well_fov}.parquet` — organoid features (required)
- `nucleocentric_profiles_{well_fov}.parquet` — nucleocentric features (optional --
  treated as empty if this file doesn't exist, since not every pipeline that feeds
  this script produces deep-learning Nucleocentric data)

## Outputs
Three enriched parquet files written to `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`:

| File | Added columns |
|---|---|
| `sc_profiles_{well_fov}_related.parquet` | `ParentOrganoid`, shell/distance features |
| `organoid_profiles_{well_fov}_related.parquet` | `OrganoidSingleCellCount` |
| `nucleocentric_profiles_{well_fov}_related.parquet` | `ParentOrganoid` |

## Notes
- Parent organoid assignment uses bbox containment: a cell is assigned to the first
  organoid whose bounding box contains the cell's nuclear centroid.
- Spatial features are computed using Mahalanobis distance with a regularized covariance
  matrix (applied automatically when cell count is low).
- Shell classification divides cells into 4 concentric shells from organoid centroid
  outward, requiring at least 3 cells per shell.
- Object identifiers: accepts either the older CellProfiler-era pipeline's own
  `object_id` column (produced by IBP steps 00/0a/1/2) or ZEDProfiler's native
  `Metadata_Object_ObjectID`, normalized to `object_id` right after loading (see
  the data-loading cell below) rather than requiring a caller to rename it first.
  `image_set` is set directly from this script's own `well_fov` argument rather
  than required as an input column, since every row in a given input file
  belongs to the single well-FOV this script is invoked for.
- `nucleocentric_profiles_{well_fov}.parquet` is optional: if it doesn't exist,
  an empty dataframe is used instead of requiring a caller to manufacture a
  placeholder file. The output nucleocentric_profiles_{well_fov}_related.parquet
  is still always written (empty, if the input was empty) for schema consistency
  with the other two output files.

In [1]:
import os
import pathlib

import numpy as np
import pandas as pd
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.neighbors_utils import (
    classify_cells_into_shells,
    euclidean_distance_from_centroid,
    mahalanobis_distance_from_centroid,
)
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm import tqdm

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C10-1"
    image_based_profiles_subparent_name = "image_based_profiles"

### Pathing

In [3]:
# input paths
sc_profile_handcrafted_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_handcrafted_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve(strict=True)

sc_profile_sammed_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_sammed_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_sammed_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_sammed_profiles_{well_fov}.parquet"
).resolve(strict=True)
# Not strict=True like the other two inputs: ZEDProfiler produces no
# Nucleocentric (deep-learning) features at all, so a caller without any
# real Nucleocentric data has nothing to put here -- see the loading cell
# below, which falls back to an empty dataframe when this file is absent
# rather than requiring every caller to manufacture a placeholder file.
nucleocentric_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
# output paths
sc_profile_handcrafted_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_related.parquet"
).resolve()
organoid_profile_handcrafted_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_sammed_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_sammed_related.parquet"
).resolve()
organoid_profile_sammed_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_sammed_related.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/nucleocentric_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_handcrafted_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
sc_profile_df = pd.read_parquet(sc_profile_handcrafted_path)
organoid_profile_df = pd.read_parquet(organoid_profile_handcrafted_path)
sc_profile_sammed_df = pd.read_parquet(sc_profile_sammed_path)
organoid_profile_sammed_df = pd.read_parquet(organoid_profile_sammed_path)
# ZEDProfiler-fed callers have no real Nucleocentric data to provide --
# fall back to an empty frame rather than requiring one. object_id is
# set here so the merge further down (which joins on object_id +
# image_set) has something to join against; image_set is set
# unconditionally for both dataframes just below regardless.
if nucleocentric_profile_path.exists():
    nucleocentric_df = pd.read_parquet(nucleocentric_profile_path)
else:
    nucleocentric_df = pd.DataFrame(columns=["object_id"])

# Normalize the object identifier column name: ZEDProfiler's own
# Metadata_Object_ObjectID is accepted directly, same object concept as
# the older CellProfiler-era pipeline's own `object_id` -- renamed once,
# here, rather than requiring a caller to disguise ZEDProfiler's column
# as the older convention before handoff. A no-op wherever `object_id`
# is already present.
for _df in (sc_profile_df, organoid_profile_df, nucleocentric_df):
    if "object_id" not in _df.columns and "Metadata_Object_ObjectID" in _df.columns:
        _df.rename(columns={"Metadata_Object_ObjectID": "object_id"}, inplace=True)

# `image_set` is just this well-FOV's own label -- this script already
# has it as `well_fov`, so set it directly rather than requiring it as an
# input column. Both dataframes are scoped to this single well-FOV
# already, so every row gets the same value.
sc_profile_df["image_set"] = well_fov
nucleocentric_df["image_set"] = well_fov

print(f"Single-cell profile shape: {sc_profile_df.shape}")
print(f"Organoid profile shape: {organoid_profile_df.shape}")
print(f"Single-cell sammed profile shape: {sc_profile_sammed_df.shape}")
print(f"Organoid sammed profile shape: {organoid_profile_sammed_df.shape}")
print(f"Nucleocentric profile shape: {nucleocentric_df.shape}")

Single-cell profile shape: (13, 2676)
Organoid profile shape: (1, 903)
Single-cell sammed profile shape: (13, 9218)
Organoid sammed profile shape: (1, 3074)
Nucleocentric profile shape: (13, 3074)


In [5]:
x_y_z_sc_colnames = [
    x
    for x in sc_profile_df.columns
    # "area": CellProfiler-era naming (*_AreaSizeShape_*). "volumesizeshape":
    # ZEDProfiler's own naming (*_VolumeSizeShape_*) for the same measurement
    # family -- accepted directly so ZEDProfiler-sourced data needs no
    # column renaming to work with this notebook.
    if ("area" in x.lower() or "volumesizeshape" in x.lower())
    and "center" in x.lower()
    and "nuclei" in x.lower()
]
x_y_z_sc_colnames

['Nuclei_NoChannel_VolumeSizeShape_CenterX',
 'Nuclei_NoChannel_VolumeSizeShape_CenterY',
 'Nuclei_NoChannel_VolumeSizeShape_CenterZ']

In [6]:
organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    # "area" (CellProfiler) or "volumesizeshape" (ZEDProfiler) -- see the
    # x_y_z_sc_colnames cell above for why both are accepted.
    if ("area" in x.lower() or "volumesizeshape" in x.lower())
    and ("min" in x.lower() or "max" in x.lower())
]
organoid_bbox_colnames = sorted(organoid_bbox_colnames)

# When sorted alphabetically, the bbox column names fall in this order:
#   [0] = *MaxX, [1] = *MaxY, [2] = *MaxZ, [3] = *MinX, [4] = *MinY, [5] = *MinZ
# This ordering is assumed in the bbox tuple construction below. Holds
# regardless of whether the matched family is AreaSizeShape or
# VolumeSizeShape: the family name prefix is identical across all six
# candidates, so sort order is determined only by the trailing Max/Min +
# axis letter.

In [7]:
# Initialize ParentOrganoid to -1 (sentinel for unassigned cells).
sc_profile_df["ParentOrganoid"] = -1

# Extract single-cell centroids as numpy array for faster access
sc_centroids = sc_profile_df[x_y_z_sc_colnames].values  # (N_cells, 3) array

# reshape the centroids to be in z,y,x order for easier comparison with bbox
sc_centroids = sc_centroids[:, [2, 1, 0]]

# Loop through organoids with progress bar
for organoid_index, organoid_row in tqdm(
    organoid_profile_df.iterrows(),
    total=len(organoid_profile_df),
    desc="Assigning cells to organoids",
):
    # Build the bbox tuple using the sorted column order documented in the cell above:
    # sorted alphabetically gives [MaxX, MaxY, MaxZ, MinX, MinY, MinZ]
    # so indices [5]=MinZ, [4]=MinY, [3]=MinX, [2]=MaxZ, [1]=MaxY, [0]=MaxX.
    organoid_bbox = (
        organoid_row[organoid_bbox_colnames[5]],  # z_min
        organoid_row[organoid_bbox_colnames[4]],  # y_min
        organoid_row[organoid_bbox_colnames[3]],  # x_min
        organoid_row[organoid_bbox_colnames[2]],  # z_max
        organoid_row[organoid_bbox_colnames[1]],  # y_max
        organoid_row[organoid_bbox_colnames[0]],  # x_max
    )

    z_min, y_min, x_min, z_max, y_max, x_max = organoid_bbox

    # Vectorized bbox check - much faster!
    mask = (
        (sc_centroids[:, 0] >= z_min)  # z
        & (sc_centroids[:, 0] <= z_max)  # z
        & (sc_centroids[:, 1] >= y_min)
        & (sc_centroids[:, 1] <= y_max)
        & (sc_centroids[:, 2] >= x_min)
        & (sc_centroids[:, 2] <= x_max)
    )

    # First-match-wins: if organoid bboxes overlap, a cell is assigned to the first
    # organoid whose bbox contains it and is never reassigned to a later one.
    # Both masks are NumPy arrays (positional) to avoid pandas index misalignment.
    unassigned_mask = sc_profile_df["ParentOrganoid"].values == -1
    final_mask = mask & unassigned_mask

    # Assign parent organoid to matching cells
    sc_profile_df.loc[sc_profile_df.index[final_mask], "ParentOrganoid"] = organoid_row[
        "object_id"
    ]

print(f"Assigned {(sc_profile_df['ParentOrganoid'] != -1).sum()} cells to organoids")
print(f"Unassigned cells: {(sc_profile_df['ParentOrganoid'] == -1).sum()}")

Assigning cells to organoids: 100%|██████████| 1/1 [00:00<00:00, 377.08it/s]

Assigned 11 cells to organoids
Unassigned cells: 2


### Add single-cell counts for each organoid

In [8]:
organoid_sc_counts = (
    sc_profile_df["ParentOrganoid"]
    .value_counts()
    .to_frame(name="OrganoidSingleCellCount")
    .reset_index()
)
# merge the organoid profile with the single-cell counts
organoid_profile_df = pd.merge(
    organoid_profile_df,
    organoid_sc_counts,
    left_on="object_id",
    right_on="ParentOrganoid",
    how="left",
).drop(columns=["ParentOrganoid"])
sc_count = organoid_profile_df.pop("OrganoidSingleCellCount")
organoid_profile_df.insert(2, "OrganoidSingleCellCount", sc_count)

### Carry the `ParentOrganoid` assignment and spatial features over to the sammed profiles, so that the same cells are assigned to the same organoids in both the

In [9]:
organoid_profile_sammed_df = organoid_profile_sammed_df.merge(
    organoid_profile_df[["object_id", "OrganoidSingleCellCount"]],
    left_on="object_id",
    right_on="object_id",
    how="left",
)

In [10]:
sc_profile_sammed_df = sc_profile_sammed_df.merge(
    sc_profile_df[["object_id", "ParentOrganoid"]],
    left_on="object_id",
    right_on="object_id",
    how="left",
)

### Empty dataframe fallbacks

If either the organoid or SC profile is empty for this well-FOV, a placeholder row
is inserted so that downstream merges always find consistent columns.

In [11]:
# replace NaN with 0 for organoids that have no assigned cells
organoid_profile_df["OrganoidSingleCellCount"] = (
    organoid_profile_df["OrganoidSingleCellCount"].fillna(0).astype(int)
)
organoid_profile_df.head()

,Metadata_Compartment,Metadata_Segmentation_PrimaryChannel,OrganoidSingleCellCount,Metadata_Segmentation_PrimaryChannelCode,Metadata_Segmentation_SeedChannel,Metadata_Segmentation_SeedChannelCode,Metadata_Segmentation_Method,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,...,Organoid_ER-Mito_Colocalization_RankWeightedColocalizationCoeff1,Organoid_ER-Mito_Colocalization_RankWeightedColocalizationCoeff2,Organoid_AGP-Mito_Colocalization_Correlation,Organoid_AGP-Mito_Colocalization_MandersCoeffM1,Organoid_AGP-Mito_Colocalization_MandersCoeffM2,Organoid_AGP-Mito_Colocalization_OverlapCoeff,Organoid_AGP-Mito_Colocalization_MandersCoeffCostesM1,Organoid_AGP-Mito_Colocalization_MandersCoeffCostesM2,Organoid_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff1,Organoid_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff2
0,Organoid,AGP,11,555,,,segmented_from_agp,NF0014_T1,NF0014,NF0014_T1,...,0.752368,0.77318,0.820858,0.99953,0.781476,87.802307,1.0,0.999998,0.862535,0.861189


In [12]:
if organoid_profile_df.empty:
    # Write the empty DataFrame as-is. Parquet preserves schema (columns) even with
    # zero rows, so downstream union_by_name in 5.combining_profiles handles this
    # correctly. A fake zero-filled row was previously inserted here but produced
    # object_id=0 (the background label), creating a spurious organoid row that
    # propagated through all downstream stages.
    pass

In [13]:
print(f"Single-cell profile shape: {sc_profile_df.shape}")

Single-cell profile shape: (13, 2677)


In [14]:
if sc_profile_df.empty:
    # add a row with Na values
    sc_profile_df.loc[len(sc_profile_df)] = [None] * len(sc_profile_df.columns)
    sc_profile_df["image_set"] = well_fov

In [15]:
nucleocentric_df

,object_id,image_set,Nucleocentric_ER_SAMMed3D_Feature0,Nucleocentric_ER_SAMMed3D_Feature1,Nucleocentric_ER_SAMMed3D_Feature10,Nucleocentric_ER_SAMMed3D_Feature100,Nucleocentric_ER_SAMMed3D_Feature101,Nucleocentric_ER_SAMMed3D_Feature102,Nucleocentric_ER_SAMMed3D_Feature103,Nucleocentric_ER_SAMMed3D_Feature104,...,Nucleocentric_DNA_CHAMMI75_Feature90,Nucleocentric_DNA_CHAMMI75_Feature91,Nucleocentric_DNA_CHAMMI75_Feature92,Nucleocentric_DNA_CHAMMI75_Feature93,Nucleocentric_DNA_CHAMMI75_Feature94,Nucleocentric_DNA_CHAMMI75_Feature95,Nucleocentric_DNA_CHAMMI75_Feature96,Nucleocentric_DNA_CHAMMI75_Feature97,Nucleocentric_DNA_CHAMMI75_Feature98,Nucleocentric_DNA_CHAMMI75_Feature99
0,257,C10-1,-0.387588,-0.331411,0.263937,-0.023024,-0.166596,0.133375,0.009691,-0.041984,...,0.741046,2.473222,-4.719810,3.142038,1.414180,0.171295,-2.359238,5.339288,-0.032572,1.317237
1,514,C10-1,-0.520189,-0.267505,0.189327,-0.046270,-0.095591,0.032853,-0.076846,-0.029667,...,2.327991,3.689584,-0.889327,0.776212,-6.122406,-2.416398,-0.493841,3.049184,3.919286,-5.151403
2,771,C10-1,-0.581641,-0.219237,0.182019,-0.031128,-0.043808,0.087422,-0.110645,0.001540,...,5.739287,8.016791,-4.434169,-2.146059,0.676049,-4.159507,3.135860,4.350221,3.436403,-2.668257
3,1028,C10-1,-0.449190,-0.264845,0.250450,-0.015267,-0.111339,0.157119,0.006628,-0.105172,...,3.705687,-0.001241,-1.421754,-2.984232,-5.115265,-3.466706,1.138514,0.556295,-0.036136,-5.796140
4,1285,C10-1,0.103962,-0.054048,0.238278,-0.019080,-0.038646,0.208953,0.103333,-0.250066,...,3.839004,5.565547,-3.040703,0.474499,-2.339533,-3.483905,-0.563513,3.294501,3.468483,-6.326318
5,1542,C10-1,-0.148116,-0.178462,0.004337,-0.100847,0.003336,0.258022,0.089748,-0.233933,...,2.904614,6.272840,-7.120878,-2.095264,-0.129440,-7.946099,3.286433,1.447455,-0.067315,-3.884610
6,1799,C10-1,-0.037031,-0.120660,0.083667,-0.190462,0.057396,0.192158,0.033739,-0.153454,...,-0.543002,1.574917,-4.177020,-0.594166,-0.335635,-4.639916,-0.983635,1.976023,3.100438,-5.522274
7,2056,C10-1,0.030130,-0.179593,0.085482,-0.023347,-0.050832,0.356853,0.057691,-0.162994,...,6.351715,6.173274,-8.588425,-1.166324,-4.443715,-5.096104,-1.885387,1.358625,-1.037977,-4.188878
8,2313,C10-1,0.057216,-0.171916,0.090158,-0.082445,0.037093,0.275191,0.145966,-0.207962,...,0.821564,2.306137,-10.785554,-1.453985,2.634078,-5.566370,2.677842,1.384742,-2.473828,2.302386
9,2570,C10-1,-0.099754,-0.210529,0.096264,-0.092254,-0.137732,0.381143,0.094719,-0.136803,...,1.938287,-0.229218,-7.541558,0.895820,-1.198673,-4.692612,3.067567,0.518035,-1.960302,-3.124177


In [16]:
# Propagate ParentOrganoid to nucleocentric profiles.
# Nucleocentric objects share object_id with their parent nucleus, so joining
# on object_id + image_set carries the organoid assignment through.
nucleocentric_df = pd.merge(
    nucleocentric_df,
    sc_profile_df[["object_id", "image_set", "ParentOrganoid"]],
    on=["object_id", "image_set"],
    how="left",
)

## Get single cell and organoid relationships and spatial distributions

In [17]:
x_y_z_organoid_centroid_colnames = [
    x
    for x in organoid_profile_df.columns
    # "area" (CellProfiler) or "volumesizeshape" (ZEDProfiler) -- see the
    # x_y_z_sc_colnames cell above for why both are accepted.
    if ("area" in x.lower() or "volumesizeshape" in x.lower()) and "center" in x.lower()
]
x_y_z_organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if ("area" in x.lower() or "volumesizeshape" in x.lower())
    and ("min" in x.lower() or "max" in x.lower())
]

In [18]:
results = []

# get the organoid id and the single-cells for each

organoid_ids = organoid_profile_df["object_id"]

# organoid_id = organoid_ids[0]

for organoid_id in organoid_ids:
    organoid_centroid = (
        organoid_profile_df.loc[
            organoid_profile_df["object_id"] == organoid_id,
            x_y_z_organoid_centroid_colnames,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
        .to_numpy(dtype=float)
    )
    organoid_bbox = organoid_profile_df.loc[
        organoid_profile_df["object_id"] == organoid_id, x_y_z_organoid_bbox_colnames
    ].values[0]
    single_cells_in_organoid = sc_profile_df[
        sc_profile_df["ParentOrganoid"] == organoid_id
    ]
    if single_cells_in_organoid.empty:
        print(f"No single cells assigned to organoid {organoid_id}")
        continue

    single_cells_centroids = (
        single_cells_in_organoid[x_y_z_sc_colnames]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_rows = ~pd.isna(single_cells_centroids).any(axis=1)
    single_cells_centroids = single_cells_centroids[valid_rows]
    single_cells_in_organoid = single_cells_in_organoid.loc[valid_rows]

    if single_cells_centroids.shape[0] == 0:
        continue

    # convert to a dict with the key being the object_id
    # rename the centroids to z,y.x
    # x_y_z_sc_colnames is alphabetically sorted: [CenterX, CenterY, CenterZ]
    # so index [0]=X, [1]=Y, [2]=Z. The dict remaps them to named z/y/x keys.
    single_cells_centroids_dict = {
        "object_id": single_cells_in_organoid["object_id"].to_numpy(),
        "z": single_cells_in_organoid[x_y_z_sc_colnames[2]].to_numpy(dtype=float),
        "y": single_cells_in_organoid[x_y_z_sc_colnames[1]].to_numpy(dtype=float),
        "x": single_cells_in_organoid[x_y_z_sc_colnames[0]].to_numpy(dtype=float),
    }

    euclidean_distance = euclidean_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    mahalanobis_distance = mahalanobis_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    shell_classification, centroid = classify_cells_into_shells(
        coords=single_cells_centroids_dict,
        n_shells=4,
        method="mahalanobis",
        min_cells_per_shell=3,
        centroid=organoid_centroid,
    )

    shell_classification_df = pd.DataFrame(shell_classification)
    shell_classification_df["ParentOrganoid"] = organoid_id
    results.append(shell_classification_df)

           Reducing to 3 shells for statistical reliability


In [19]:
# Concatenate per-organoid shell results and rename columns to standard feature name format.
# Added columns (under Nuclei_NoChannel_Neighbors_*):
#   ShellAssignments            — shell index (1=innermost, N=outermost) for each cell
#   DistancesFromCenter         — Mahalanobis distance from organoid centroid
#   DistancesFromExterior       — distance from the outermost shell boundary
#   NormalizedDistancesFromCenter — DistancesFromCenter normalized to [0, 1]
#   ShellsUsed                  — total number of shells actually assigned (may be < 4
#                                 if too few cells to fill all shells)
if results:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

else:
    df = pd.DataFrame(columns=["object_id", "ParentOrganoid"])

# rename the columns

df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment="Nuclei",
            feature_type="Neighbors",
            channel="NoChannel",
            measurement=col,
        )
        for col in df.columns
        if col not in ["object_id", "ParentOrganoid"]
    },
    inplace=True,
)

In [20]:
# concat the shell classification with the single cell profile df to get the full single cell profile with the shell classification and the parent organoid id
sc_profile_with_shells_df = pd.merge(
    sc_profile_df,
    df,
    left_on=["object_id", "ParentOrganoid"],
    right_on=["object_id", "ParentOrganoid"],
    how="left",
)

### Save the profiles

In [21]:
organoid_profile_df.to_parquet(organoid_profile_handcrafted_output_path, index=False)
organoid_profile_df.head()

,Metadata_Compartment,Metadata_Segmentation_PrimaryChannel,OrganoidSingleCellCount,Metadata_Segmentation_PrimaryChannelCode,Metadata_Segmentation_SeedChannel,Metadata_Segmentation_SeedChannelCode,Metadata_Segmentation_Method,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,...,Organoid_ER-Mito_Colocalization_RankWeightedColocalizationCoeff1,Organoid_ER-Mito_Colocalization_RankWeightedColocalizationCoeff2,Organoid_AGP-Mito_Colocalization_Correlation,Organoid_AGP-Mito_Colocalization_MandersCoeffM1,Organoid_AGP-Mito_Colocalization_MandersCoeffM2,Organoid_AGP-Mito_Colocalization_OverlapCoeff,Organoid_AGP-Mito_Colocalization_MandersCoeffCostesM1,Organoid_AGP-Mito_Colocalization_MandersCoeffCostesM2,Organoid_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff1,Organoid_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff2
0,Organoid,AGP,11,555,,,segmented_from_agp,NF0014_T1,NF0014,NF0014_T1,...,0.752368,0.77318,0.820858,0.99953,0.781476,87.802307,1.0,0.999998,0.862535,0.861189


In [22]:
sc_profile_with_shells_df.to_parquet(sc_profile_handcrafted_output_path, index=False)
sc_profile_with_shells_df.head()

,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,Metadata_Experiment_ImageSet,object_id,Nuclei_DNA_Granularity_1,Nuclei_DNA_Granularity_2,...,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM2,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff1,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff2,image_set,ParentOrganoid,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellsUsed
0,NF0014_T1,NF0014,NF0014_T1,C10,1,NF0014_T1__NF0014_T1__C10__F1,NF0014_T1__NF0014_T1__C10__F1,1,59.963964,1.966538,...,0.971811,0.882018,0.912934,C10-1,-1,NaN,NaN,NaN,NaN,NaN
1,NF0014_T1,NF0014,NF0014_T1,C10,1,NF0014_T1__NF0014_T1__C10__F1,NF0014_T1__NF0014_T1__C10__F1,2,30.277436,1.904758,...,0.999984,0.852430,0.849885,C10-1,1,1.0,143.852716,130.890272,0.523590,3.0
2,NF0014_T1,NF0014,NF0014_T1,C10,1,NF0014_T1__NF0014_T1__C10__F1,NF0014_T1__NF0014_T1__C10__F1,3,22.004656,2.769721,...,0.999979,0.878041,0.874144,C10-1,1,1.0,168.433865,106.309123,0.613060,3.0
3,NF0014_T1,NF0014,NF0014_T1,C10,1,NF0014_T1__NF0014_T1__C10__F1,NF0014_T1__NF0014_T1__C10__F1,4,26.118181,1.331152,...,0.706807,0.772057,0.766326,C10-1,-1,NaN,NaN,NaN,NaN,NaN
4,NF0014_T1,NF0014,NF0014_T1,C10,1,NF0014_T1__NF0014_T1__C10__F1,NF0014_T1__NF0014_T1__C10__F1,5,25.937420,2.162997,...,0.999656,0.827753,0.820565,C10-1,1,1.0,128.051614,146.691374,0.466078,3.0


In [23]:
nucleocentric_df.to_parquet(nucleocentric_profile_output_path, index=False)
nucleocentric_df.head()

,object_id,image_set,Nucleocentric_ER_SAMMed3D_Feature0,Nucleocentric_ER_SAMMed3D_Feature1,Nucleocentric_ER_SAMMed3D_Feature10,Nucleocentric_ER_SAMMed3D_Feature100,Nucleocentric_ER_SAMMed3D_Feature101,Nucleocentric_ER_SAMMed3D_Feature102,Nucleocentric_ER_SAMMed3D_Feature103,Nucleocentric_ER_SAMMed3D_Feature104,...,Nucleocentric_DNA_CHAMMI75_Feature91,Nucleocentric_DNA_CHAMMI75_Feature92,Nucleocentric_DNA_CHAMMI75_Feature93,Nucleocentric_DNA_CHAMMI75_Feature94,Nucleocentric_DNA_CHAMMI75_Feature95,Nucleocentric_DNA_CHAMMI75_Feature96,Nucleocentric_DNA_CHAMMI75_Feature97,Nucleocentric_DNA_CHAMMI75_Feature98,Nucleocentric_DNA_CHAMMI75_Feature99,ParentOrganoid
0,257,C10-1,-0.387588,-0.331411,0.263937,-0.023024,-0.166596,0.133375,0.009691,-0.041984,...,2.473222,-4.719810,3.142038,1.414180,0.171295,-2.359238,5.339288,-0.032572,1.317237,NaN
1,514,C10-1,-0.520189,-0.267505,0.189327,-0.046270,-0.095591,0.032853,-0.076846,-0.029667,...,3.689584,-0.889327,0.776212,-6.122406,-2.416398,-0.493841,3.049184,3.919286,-5.151403,NaN
2,771,C10-1,-0.581641,-0.219237,0.182019,-0.031128,-0.043808,0.087422,-0.110645,0.001540,...,8.016791,-4.434169,-2.146059,0.676049,-4.159507,3.135860,4.350221,3.436403,-2.668257,NaN
3,1028,C10-1,-0.449190,-0.264845,0.250450,-0.015267,-0.111339,0.157119,0.006628,-0.105172,...,-0.001241,-1.421754,-2.984232,-5.115265,-3.466706,1.138514,0.556295,-0.036136,-5.796140,NaN
4,1285,C10-1,0.103962,-0.054048,0.238278,-0.019080,-0.038646,0.208953,0.103333,-0.250066,...,5.565547,-3.040703,0.474499,-2.339533,-3.483905,-0.563513,3.294501,3.468483,-6.326318,NaN


In [24]:
sc_profile_sammed_df.to_parquet(sc_profile_sammed_output_path, index=False)
sc_profile_sammed_df.head()

,Nuclei_ER_SAMMed3D_Feature-cls0,Nuclei_ER_SAMMed3D_Feature-cls1,Nuclei_ER_SAMMed3D_Feature-cls10,Nuclei_ER_SAMMed3D_Feature-cls100,Nuclei_ER_SAMMed3D_Feature-cls101,Nuclei_ER_SAMMed3D_Feature-cls102,Nuclei_ER_SAMMed3D_Feature-cls103,Nuclei_ER_SAMMed3D_Feature-cls104,Nuclei_ER_SAMMed3D_Feature-cls105,Nuclei_ER_SAMMed3D_Feature-cls106,...,Cytoplasm_ER_SAMMed3D_Feature-global93,Cytoplasm_ER_SAMMed3D_Feature-global94,Cytoplasm_ER_SAMMed3D_Feature-global95,Cytoplasm_ER_SAMMed3D_Feature-global96,Cytoplasm_ER_SAMMed3D_Feature-global97,Cytoplasm_ER_SAMMed3D_Feature-global98,Cytoplasm_ER_SAMMed3D_Feature-global99,object_id,image_set,ParentOrganoid
0,-0.227420,-0.295441,0.244045,0.000869,-0.163549,0.174686,0.042360,-0.095115,-0.218127,-0.042278,...,-0.010962,0.053600,-0.035529,0.168429,0.021584,0.288965,0.035805,257,C10-1,NaN
1,-0.227117,-0.317473,0.255748,0.003300,-0.176931,0.165089,0.049506,-0.090514,-0.210043,-0.028593,...,-0.010964,0.040114,0.058079,0.135199,0.032364,0.155329,0.135622,514,C10-1,NaN
2,-0.248314,-0.310342,0.252507,-0.000917,-0.170225,0.165833,0.039453,-0.089732,-0.220111,-0.040254,...,-0.010982,0.039758,0.024901,0.149033,0.033102,0.209460,0.092910,771,C10-1,NaN
3,-0.223603,-0.295035,0.245586,0.003124,-0.163309,0.172293,0.045494,-0.093141,-0.216246,-0.040277,...,-0.010977,0.053757,-0.038288,0.169039,0.023110,0.291139,0.030337,1028,C10-1,NaN
4,-0.229351,-0.302773,0.252493,-0.001763,-0.162583,0.163602,0.048340,-0.090331,-0.212466,-0.030728,...,-0.011139,0.053631,-0.025307,0.163485,0.025418,0.277753,0.036320,1285,C10-1,NaN


In [25]:
organoid_profile_sammed_df.to_parquet(organoid_profile_sammed_output_path, index=False)
organoid_profile_sammed_df.head()

,Organoid_DNA_SAMMed3D_Feature-cls0,Organoid_DNA_SAMMed3D_Feature-cls1,Organoid_DNA_SAMMed3D_Feature-cls10,Organoid_DNA_SAMMed3D_Feature-cls100,Organoid_DNA_SAMMed3D_Feature-cls101,Organoid_DNA_SAMMed3D_Feature-cls102,Organoid_DNA_SAMMed3D_Feature-cls103,Organoid_DNA_SAMMed3D_Feature-cls104,Organoid_DNA_SAMMed3D_Feature-cls105,Organoid_DNA_SAMMed3D_Feature-cls106,...,Organoid_Mito_SAMMed3D_Feature-global93,Organoid_Mito_SAMMed3D_Feature-global94,Organoid_Mito_SAMMed3D_Feature-global95,Organoid_Mito_SAMMed3D_Feature-global96,Organoid_Mito_SAMMed3D_Feature-global97,Organoid_Mito_SAMMed3D_Feature-global98,Organoid_Mito_SAMMed3D_Feature-global99,object_id,image_set,OrganoidSingleCellCount
0,-0.378976,-0.34836,0.269897,-0.000645,-0.214004,0.091996,0.045562,-0.047904,-0.223609,-0.023807,...,-0.010664,0.041979,0.011682,0.084084,0.034811,0.179321,0.114369,1,C10-1,11
